# Batch 03: position, grid and coastal geometry
Read-only review of 13 fields. No data modification, imputation, snapping, feature selection or test access.
Full development coverage for stored-column completeness, geometry predicates, R membership, cells and visit definitions. Mask recomputation covers all R rows, all globally flagged geometry rows and trajectory endpoints.
Run using the project environment from work_place or a child directory. Counts describe observations, not qualified future endpoint pairs.

In [1]:
from pathlib import Path
import json,hashlib
import numpy as np
import pyarrow.parquet as pq
import shapely as sh
from shapely.geometry import shape
from data3001.dataset import development_ids,track_path
from data3001.paths import project_paths
root=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'configs/dataset_manifest.json').exists())
paths=project_paths();manifest=json.loads((root/'configs/dataset_manifest.json').read_text())
assert hashlib.sha256((root/'docs/contract/data_contract_v1.0.xlsx').read_bytes()).hexdigest()==manifest['contract_sha256']
Rpath=paths.asset('release_R',verify=True)
R=shape(json.loads(Rpath.read_text())['features'][0]['geometry']);sh.prepare(R)
mask_paths={k:paths.asset(key,verify=True) for k,key in [('h','land_mask_high'),('i','land_mask_intermediate')]}
land={k:sh.from_wkb(path.read_bytes()) for k,path in mask_paths.items()}
for g in land.values():sh.prepare(g)
fields='lat lon lon_normalized position_ok in_R wet_high coast_resolution_disagreement mapping_ok geometry_uncertain cell_1 cell_05 visit_cell_1 visit_cell_05'.split()
nulls={k:0 for k in fields}
checks={k:0 for k in ['nonfinite_or_out_of_range_coordinates','normalized_longitude_mismatch','position_ok_mismatch','R_membership_mismatch','mapping_predicate_mismatch','uncertainty_predicate_mismatch','cell_1_mismatch','cell_05_mismatch','nested_grid_parent_mismatch','visit_cell_1_mismatch','visit_cell_05_mismatch','high_mask_recheck_mismatch','coast_disagreement_recheck_mismatch']}
counts={k:0 for k in ['IDs','rows','R_rows','R_IDs','position_ok_false','wet_high_false','coast_disagreement','mapping_ok_false','geometry_uncertain','R_mapping_ok_false','R_coast_disagreement','R_wet_high_false','longitude_representation_differences','mask_rechecked_rows','mask_rechecked_R_rows','R_unique_cells_1','R_unique_cells_05']}
geom_IDs=set();grid_R={1:set(),.5:set()};examples=[];max_lon_delta=0.0
cross={'high_land_and_disagree':0,'high_land_and_agree':0,'high_water_and_disagree':0}
for ident in development_ids():
    tab=pq.read_table(track_path(ident),columns=fields+['time','analytical_row','source_obs','err_lat','err_lon'])
    a={k:tab[k].to_numpy() for k in fields+['time','analytical_row','source_obs','err_lat','err_lon']}
    n=tab.num_rows;assert n==manifest['records'][str(ident)]['rows']
    for k in fields:nulls[k]+=tab[k].null_count
    lon=a['lon'];lat=a['lat'];t=a['time'];norm=a['lon_normalized'];inside=a['in_R'];pos=a['position_ok']
    coord=np.isfinite(lon)&np.isfinite(lat)&(lon>=-180)&(lon<=180)&(lat>=-90)&(lat<=90)
    expected_norm=(lon+180)%360-180
    counts['IDs']+=1;counts['rows']+=n;counts['R_rows']+=int(inside.sum());counts['R_IDs']+=int(inside.any())
    checks['nonfinite_or_out_of_range_coordinates']+=int((~coord).sum())
    checks['normalized_longitude_mismatch']+=int((norm!=expected_norm).sum())
    checks['position_ok_mismatch']+=int((pos!=(coord&a['analytical_row'])).sum())
    expected_R=sh.intersects_xy(R,norm,lat)&pos
    checks['R_membership_mismatch']+=int((inside!=expected_R).sum())
    checks['mapping_predicate_mismatch']+=int((a['mapping_ok']!=(pos&a['wet_high']&~a['coast_resolution_disagreement'])).sum())
    checks['uncertainty_predicate_mismatch']+=int((a['geometry_uncertain']!=(pos&~a['mapping_ok'])).sum())
    for field,key in [('position_ok','position_ok_false'),('wet_high','wet_high_false'),('mapping_ok','mapping_ok_false')]:counts[key]+=int((~a[field]).sum())
    counts['coast_disagreement']+=int(a['coast_resolution_disagreement'].sum());counts['geometry_uncertain']+=int(a['geometry_uncertain'].sum())
    counts['R_mapping_ok_false']+=int((inside&~a['mapping_ok']).sum());counts['R_coast_disagreement']+=int((inside&a['coast_resolution_disagreement']).sum());counts['R_wet_high_false']+=int((inside&~a['wet_high']).sum())
    counts['longitude_representation_differences']+=int((norm!=lon).sum());max_lon_delta=max(max_lon_delta,float(np.max(np.abs(norm-lon))))
    bad=a['geometry_uncertain']
    if bad.any():geom_IDs.add(ident)
    cross['high_land_and_disagree']+=int((~a['wet_high']&a['coast_resolution_disagreement']).sum())
    cross['high_land_and_agree']+=int((~a['wet_high']&~a['coast_resolution_disagreement']).sum())
    cross['high_water_and_disagree']+=int((a['wet_high']&a['coast_resolution_disagreement']).sum())
    for res,cell_name,visit_name in [(1,'cell_1','visit_cell_1'),(.5,'cell_05','visit_cell_05')]:
        nx=int(360/res);valid=pos&(lat<90)
        expected_cell=np.full(n,-1,dtype=np.int64)
        expected_cell[valid]=np.floor((lat[valid]+90)/res).astype(np.int64)*nx+np.floor(((norm[valid]+180)%360)/res).astype(np.int64)
        cell=a[cell_name]
        checks[cell_name+'_mismatch']+=int((cell!=expected_cell).sum())
        new=np.r_[True,(np.diff(t)!=3600)|(np.diff(cell)!=0)|(cell[1:]<0)]
        expected_visit=np.cumsum(new)-1
        checks[visit_name+'_mismatch']+=int((a[visit_name]!=expected_visit).sum())
        grid_R[res].update(cell[inside].tolist())
    fine=a['cell_05'].astype(np.int64);parent=(fine//720//2)*360+(fine%720//2)
    checks['nested_grid_parent_mismatch']+=int((parent!=a['cell_1']).sum())
    # Recompute masks for every R observation, every globally flagged geometry row,
    # and the first/last row of every development ID; other mapped rows retain the prior full-audit evidence.
    sample=inside|bad;sample[0]=True;sample[-1]=True
    wet_h=~sh.intersects_xy(land['h'],norm[sample],lat[sample]);wet_i=~sh.intersects_xy(land['i'],norm[sample],lat[sample])
    checks['high_mask_recheck_mismatch']+=int((wet_h!=a['wet_high'][sample]).sum())
    checks['coast_disagreement_recheck_mismatch']+=int(((wet_h!=wet_i)!=a['coast_resolution_disagreement'][sample]).sum())
    counts['mask_rechecked_rows']+=int(sample.sum());counts['mask_rechecked_R_rows']+=int((sample&inside).sum())
    for j in np.flatnonzero(inside&bad):
        examples.append({'id':ident,'source_obs':int(a['source_obs'][j]),'time_UTC':str(np.datetime64(int(t[j]),'s')),'lat':float(lat[j]),'lon':float(lon[j]),'wet_high':bool(a['wet_high'][j]),'coast_resolution_disagreement':bool(a['coast_resolution_disagreement'][j]),'mapping_ok':bool(a['mapping_ok'][j]),'cell_1':int(a['cell_1'][j]),'err_lat':None if not np.isfinite(a['err_lat'][j]) else float(a['err_lat'][j]),'err_lon':None if not np.isfinite(a['err_lon'][j]) else float(a['err_lon'][j])})
assert not any(nulls.values()) and not any(checks.values())
assert counts['rows']==33943292 and counts['IDs']==3224
counts['R_unique_cells_1']=len(grid_R[1]);counts['R_unique_cells_05']=len(grid_R[.5])
result={'scope':'Development only. Final test not opened. Geometry flags are classifications, not verified instrument faults or physical beaching events.','release':manifest['version'],'contract_sha256':manifest['contract_sha256'],'null_rows_by_field':nulls,'consistency_failures':checks,'counts':counts,'geometry_uncertain_IDs':len(geom_IDs),'geometry_classification_overlap':cross,'maximum_lon_representation_delta_degrees':max_lon_delta,'R_geometry_examples':examples,'mask_recheck_scope':'Every R observation, every globally flagged geometry row, and first/last row of each ID. No claim of a fresh full-global mask recomputation.','input_file_checksums_verified':counts['IDs'],'data_modified':False,'final_model_feature_decisions_changed':False}
print(json.dumps(result,indent=2))


{
  "scope": "Development only. Final test not opened. Geometry flags are classifications, not verified instrument faults or physical beaching events.",
  "release": "per-drifter-20261016-v1",
  "contract_sha256": "6cfa91471a1399a9c3a1a8edfeb2e18cea17e0f4031ef0f8ca604ca4565a2446",
  "null_rows_by_field": {
    "lat": 0,
    "lon": 0,
    "lon_normalized": 0,
    "position_ok": 0,
    "in_R": 0,
    "wet_high": 0,
    "coast_resolution_disagreement": 0,
    "mapping_ok": 0,
    "geometry_uncertain": 0,
    "cell_1": 0,
    "cell_05": 0,
    "visit_cell_1": 0,
    "visit_cell_05": 0
  },
  "consistency_failures": {
    "nonfinite_or_out_of_range_coordinates": 0,
    "normalized_longitude_mismatch": 0,
    "position_ok_mismatch": 0,
    "R_membership_mismatch": 0,
    "mapping_predicate_mismatch": 0,
    "uncertainty_predicate_mismatch": 0,
    "cell_1_mismatch": 0,
    "cell_05_mismatch": 0,
    "nested_grid_parent_mismatch": 0,
    "visit_cell_1_mismatch": 0,
    "visit_cell_05_mismatch